# FLUX Fill BSS/BDS Condition-Anchoring Full Run

This notebook mounts Google Drive, clones or updates the lightweight experiment repo, downloads FLUX.1 Fill-dev weights to Drive only when explicitly requested, runs audit, creates synthetic fill assets/manifests, runs smoke, runs the 4-case mini-suite by default, and prints the final result summary.

Do not paste Hugging Face tokens into code cells. The token prompt below uses `getpass` and does not print the token.


In [45]:
from pathlib import Path
import os

# Set this before Run All after you push the scaffold to GitHub.
GITHUB_REPO_URL = "https://github.com/WANG-Ruipeng/FLUX-bss.git"
BRANCH = "flux-fill-bss-bds"
REPO_ROOT = Path("/content/FLUX-bss")

RUN_NAME = "flux_fill_bss_bds_v1"
RUNS_ROOT = Path("/content/FLUX-bss-Runs")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Colab_Projects/FLUX-bss")
DRIVE_RUNS_ROOT = DRIVE_PROJECT_ROOT / "runs"
DRIVE_MODELS_ROOT = DRIVE_PROJECT_ROOT / "models"
DRIVE_WEIGHTS_ROOT = DRIVE_MODELS_ROOT / "FLUX.1-Fill-dev"
EXPERIMENT_ROOT = RUNS_ROOT / RUN_NAME
DRIVE_EXPERIMENT_ROOT = DRIVE_RUNS_ROOT / RUN_NAME

INSTALL_DEPS = True
MOUNT_DRIVE = True
FORCE_RECLONE = False

# User agreed to the gated license before running this notebook.
LICENSE_ACCEPTED = True

# Default Run All uses existing Drive weights and never downloads large model files.
# Set AUTO_DOWNLOAD_IF_MISSING=True only when you intentionally want to fetch missing weights.
AUTO_DOWNLOAD_IF_MISSING = False
DOWNLOAD_WEIGHTS = False

# Full Run All: smoke stays resumable, and the 4-case mini-suite runs by default.
RUN_SMOKE = True
RUN_MINI_SUITE = True
PRINT_FINAL_SUMMARY = True
SYNC_FINAL_TO_DRIVE = True

HEIGHT = 1024
WIDTH = 1024
GUIDANCE_SCALE = 30.0
MAX_SEQUENCE_LENGTH = 512
SEED = 0
DTYPE = "bfloat16"
CPU_OFFLOAD = False

print("REPO_ROOT:", REPO_ROOT)
print("EXPERIMENT_ROOT:", EXPERIMENT_ROOT)
print("DRIVE_EXPERIMENT_ROOT:", DRIVE_EXPERIMENT_ROOT)
print("DRIVE_WEIGHTS_ROOT:", DRIVE_WEIGHTS_ROOT)
print("RUN_SMOKE:", RUN_SMOKE)
print("RUN_MINI_SUITE:", RUN_MINI_SUITE)


REPO_ROOT: /content/FLUX-bss
EXPERIMENT_ROOT: /content/FLUX-bss-Runs/flux_fill_bss_bds_v1
DRIVE_EXPERIMENT_ROOT: /content/drive/MyDrive/Colab_Projects/FLUX-bss/runs/flux_fill_bss_bds_v1
DRIVE_WEIGHTS_ROOT: /content/drive/MyDrive/Colab_Projects/FLUX-bss/models/FLUX.1-Fill-dev
RUN_SMOKE: True
RUN_MINI_SUITE: True


In [46]:
import subprocess
import sys
import time
import shutil
import getpass

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print("Drive mount skipped or failed:", repr(exc))

EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RUNS_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_MODELS_ROOT.mkdir(parents=True, exist_ok=True)
print("Runtime setup done.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runtime setup done.


In [47]:
def run(cmd, cwd=None, check=True, env=None):
    display = " ".join(str(part) for part in cmd)
    if "x-access-token:" in display:
        display = display.split("x-access-token:")[0] + "x-access-token:***@" + display.split("@", 1)[-1]
    print("$", display)
    return subprocess.run([str(part) for part in cmd], cwd=str(cwd) if cwd else None, check=check, text=True, env=env)


def show_manifest_failures(manifest_path, max_chars=4000):
    import pandas as pd

    manifest_path = Path(manifest_path)
    print(f"Inspecting manifest after failure: {manifest_path}")
    if not manifest_path.exists():
        print("manifest missing")
        return

    df = pd.read_csv(manifest_path).fillna("")
    cols = [
        col for col in [
            "run_id", "case_id", "method", "status", "error_message",
            "output_path", "schedule_json_path", "stdout_log_path", "stderr_log_path",
        ]
        if col in df.columns
    ]
    try:
        display(df[cols])
    except NameError:
        print(df[cols].to_string(index=False))

    shown = 0
    for _, row in df.iterrows():
        output_path = Path(str(row.get("output_path", "")))
        schedule_path = Path(str(row.get("schedule_json_path", "")))
        status = str(row.get("status", "")).lower()
        output_missing = bool(str(output_path)) and not output_path.exists()
        schedule_missing = bool(str(schedule_path)) and not schedule_path.exists()
        failed = status == "failed" or output_missing or schedule_missing
        if not failed:
            continue

        shown += 1
        print(f"\n=== {row.get('run_id', '')} {row.get('method', '')} ===")
        print("status:", row.get("status", ""))
        print("error:", row.get("error_message", ""))
        print("output_missing:", output_missing, output_path)
        print("schedule_missing:", schedule_missing, schedule_path)
        for key in ["stdout_log_path", "stderr_log_path"]:
            log_path = Path(str(row.get(key, "")))
            print(f"\n--- {key}: {log_path} ---")
            if log_path.exists():
                text = log_path.read_text(encoding="utf-8", errors="replace")
                print(text[-max_chars:])
            else:
                print("missing")

    if shown == 0:
        print("No failed or missing-output rows are marked in the manifest. Scroll above for subprocess output.")


def authenticated_url(url):
    if "<YOUR_ORG_OR_USER>" in url:
        return url
    token = os.environ.get("GITHUB_TOKEN", "")
    if not token:
        token = getpass.getpass("GitHub token for clone/fetch; leave blank if public: ")
        if token:
            os.environ["GITHUB_TOKEN"] = token
    if token:
        return url.replace("https://", f"https://x-access-token:{token}@", 1)
    return url


if FORCE_RECLONE and REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

if "<YOUR_ORG_OR_USER>" in GITHUB_REPO_URL and not REPO_ROOT.exists():
    raise RuntimeError("Set GITHUB_REPO_URL to your pushed repo before running in a fresh Colab runtime.")

if REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    run(["git", "-C", REPO_ROOT, "fetch", "origin", BRANCH], check=False)
    run(["git", "-C", REPO_ROOT, "checkout", BRANCH], check=False)
    run(["git", "-C", REPO_ROOT, "pull", "--ff-only", "origin", BRANCH], check=False)
elif not REPO_ROOT.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", authenticated_url(GITHUB_REPO_URL), REPO_ROOT])
    run(["git", "-C", REPO_ROOT, "remote", "set-url", "origin", GITHUB_REPO_URL], check=False)

run(["git", "-C", REPO_ROOT, "branch", "--show-current"], check=False)
run(["git", "-C", REPO_ROOT, "rev-parse", "HEAD"], check=False)
run(["git", "-C", REPO_ROOT, "status", "--short"], check=False)


$ git -C /content/FLUX-bss fetch origin flux-fill-bss-bds
$ git -C /content/FLUX-bss checkout flux-fill-bss-bds
$ git -C /content/FLUX-bss pull --ff-only origin flux-fill-bss-bds
$ git -C /content/FLUX-bss branch --show-current
$ git -C /content/FLUX-bss rev-parse HEAD
$ git -C /content/FLUX-bss status --short


CompletedProcess(args=['git', '-C', '/content/FLUX-bss', 'status', '--short'], returncode=0)

In [48]:
if INSTALL_DEPS:
    run([
        sys.executable, "-m", "pip", "install", "-U",
        "diffusers", "transformers", "accelerate", "safetensors", "sentencepiece",
        "huggingface_hub", "opencv-python", "pillow", "imageio", "pandas", "numpy", "matplotlib"
    ])


$ /usr/bin/python3 -m pip install -U diffusers transformers accelerate safetensors sentencepiece huggingface_hub opencv-python pillow imageio pandas numpy matplotlib


In [49]:
def weights_ready(path):
    path = Path(path)
    required = [
        path / "model_index.json",
        path / "scheduler",
        path / "transformer",
        path / "vae",
        path / "text_encoder",
        path / "text_encoder_2",
        path / "tokenizer",
        path / "tokenizer_2",
    ]
    if not all(item.exists() for item in required):
        return False
    return any(path.rglob("*.safetensors")) or any(path.rglob("*.bin"))


weights_are_ready = weights_ready(DRIVE_WEIGHTS_ROOT)
need_download = DOWNLOAD_WEIGHTS or (AUTO_DOWNLOAD_IF_MISSING and not weights_are_ready)
if not weights_are_ready and not need_download:
    raise RuntimeError(
        "FLUX weights are not complete at "
        f"{DRIVE_WEIGHTS_ROOT}. Run All is configured to avoid downloading large files. "
        "If you intentionally want to download them, set AUTO_DOWNLOAD_IF_MISSING=True or DOWNLOAD_WEIGHTS=True."
    )
if need_download:
    if not LICENSE_ACCEPTED:
        raise RuntimeError("Accept the FLUX.1 Fill-dev gated license before downloading weights.")
    from huggingface_hub import HfApi, login, snapshot_download
    hf_token = os.environ.get("HF_TOKEN", "") or os.environ.get("HUGGINGFACE_HUB_TOKEN", "")
    if not hf_token:
        hf_token = getpass.getpass("Hugging Face token with accepted FLUX.1 Fill-dev access: ")
    login(token=hf_token, add_to_git_credential=False)
    try:
        me = HfApi(token=hf_token).whoami()
        print("HF account:", me.get("name", "<unknown>"))
    except Exception as exc:
        print("HF token check failed before download:", repr(exc))
        raise
    DRIVE_WEIGHTS_ROOT.mkdir(parents=True, exist_ok=True)
    try:
        snapshot_download(
            repo_id="black-forest-labs/FLUX.1-Fill-dev",
            local_dir=str(DRIVE_WEIGHTS_ROOT),
            token=hf_token,
            local_files_only=False,
        )
    except Exception as exc:
        print("FLUX.1 Fill-dev download failed:", repr(exc))
        print("Check that this HF account accepted the gated model license and that the token has read access.")
        raise
else:
    print("Using existing Drive weights without download:", DRIVE_WEIGHTS_ROOT)

if not weights_ready(DRIVE_WEIGHTS_ROOT):
    raise RuntimeError(f"Weights are still missing or incomplete at {DRIVE_WEIGHTS_ROOT}")


Using existing Drive weights without download: /content/drive/MyDrive/Colab_Projects/FLUX-bss/models/FLUX.1-Fill-dev


In [50]:
env = os.environ.copy()
env["REPO_ROOT"] = str(REPO_ROOT)
env["BSS_CONDITION_REPO_DIR"] = str(REPO_ROOT)
env["EXPERIMENT_ROOT"] = str(EXPERIMENT_ROOT)
env["DRIVE_EXPERIMENT_ROOT"] = str(DRIVE_EXPERIMENT_ROOT)
env["DRIVE_WEIGHTS_ROOT"] = str(DRIVE_WEIGHTS_ROOT)

SCRIPT_ROOT = REPO_ROOT / "bss_experiments/flux_fill_bss_bds_v1/scripts"
license_flag = "yes" if LICENSE_ACCEPTED else "unknown"
audit_cmd = [
    sys.executable, SCRIPT_ROOT / "audit_flux_fill.py",
    "--run_root", EXPERIMENT_ROOT,
    "--drive_weights_root", DRIVE_WEIGHTS_ROOT,
    "--license_accepted", license_flag,
    "--dtype", DTYPE,
]
if CPU_OFFLOAD:
    audit_cmd.append("--cpu_offload")
run(audit_cmd, cwd=REPO_ROOT, env=env)


$ /usr/bin/python3 /content/FLUX-bss/bss_experiments/flux_fill_bss_bds_v1/scripts/audit_flux_fill.py --run_root /content/FLUX-bss-Runs/flux_fill_bss_bds_v1 --drive_weights_root /content/drive/MyDrive/Colab_Projects/FLUX-bss/models/FLUX.1-Fill-dev --license_accepted yes --dtype bfloat16


CompletedProcess(args=['/usr/bin/python3', '/content/FLUX-bss/bss_experiments/flux_fill_bss_bds_v1/scripts/audit_flux_fill.py', '--run_root', '/content/FLUX-bss-Runs/flux_fill_bss_bds_v1', '--drive_weights_root', '/content/drive/MyDrive/Colab_Projects/FLUX-bss/models/FLUX.1-Fill-dev', '--license_accepted', 'yes', '--dtype', 'bfloat16'], returncode=0)

In [51]:
run([
    sys.executable, SCRIPT_ROOT / "make_flux_fill_assets_and_manifest.py",
    "--run_root", EXPERIMENT_ROOT,
    "--height", HEIGHT,
    "--width", WIDTH,
    "--guidance_scale", GUIDANCE_SCALE,
    "--max_sequence_length", MAX_SEQUENCE_LENGTH,
    "--seed", SEED,
    "--dtype", DTYPE,
], cwd=REPO_ROOT, env=env)

SMOKE_MANIFEST = EXPERIMENT_ROOT / "manifests/flux_fill_smoke_manifest.csv"
MINI_MANIFEST = EXPERIMENT_ROOT / "manifests/flux_fill_mini_manifest.csv"
run([sys.executable, SCRIPT_ROOT / "validate_schedule.py", "--manifest", SMOKE_MANIFEST], cwd=REPO_ROOT, env=env)
run([sys.executable, SCRIPT_ROOT / "validate_schedule.py", "--manifest", MINI_MANIFEST], cwd=REPO_ROOT, env=env)


$ /usr/bin/python3 /content/FLUX-bss/bss_experiments/flux_fill_bss_bds_v1/scripts/make_flux_fill_assets_and_manifest.py --run_root /content/FLUX-bss-Runs/flux_fill_bss_bds_v1 --height 1024 --width 1024 --guidance_scale 30.0 --max_sequence_length 512 --seed 0 --dtype bfloat16
$ /usr/bin/python3 /content/FLUX-bss/bss_experiments/flux_fill_bss_bds_v1/scripts/validate_schedule.py --manifest /content/FLUX-bss-Runs/flux_fill_bss_bds_v1/manifests/flux_fill_smoke_manifest.csv
$ /usr/bin/python3 /content/FLUX-bss/bss_experiments/flux_fill_bss_bds_v1/scripts/validate_schedule.py --manifest /content/FLUX-bss-Runs/flux_fill_bss_bds_v1/manifests/flux_fill_mini_manifest.csv


CompletedProcess(args=['/usr/bin/python3', '/content/FLUX-bss/bss_experiments/flux_fill_bss_bds_v1/scripts/validate_schedule.py', '--manifest', '/content/FLUX-bss-Runs/flux_fill_bss_bds_v1/manifests/flux_fill_mini_manifest.csv'], returncode=0)

In [52]:
if RUN_SMOKE:
    cmd = [
        sys.executable, SCRIPT_ROOT / "run_manifest.py",
        "--manifest", SMOKE_MANIFEST,
        "--run_root", EXPERIMENT_ROOT,
        "--drive_run_root", DRIVE_EXPERIMENT_ROOT,
        "--model_path", DRIVE_WEIGHTS_ROOT,
        "--resume",
        "--sync_drive", "true",
        "--dtype", DTYPE,
    ]
    if CPU_OFFLOAD:
        cmd.append("--cpu_offload")
    try:
        run(cmd, cwd=REPO_ROOT, env=env)
    except subprocess.CalledProcessError:
        show_manifest_failures(SMOKE_MANIFEST)
        raise
    run([sys.executable, SCRIPT_ROOT / "validate_schedule.py", "--manifest", SMOKE_MANIFEST, "--require_schedule_files"], cwd=REPO_ROOT, env=env)
    run([sys.executable, SCRIPT_ROOT / "compute_metrics_against_ref.py", "--manifest", SMOKE_MANIFEST, "--run_root", EXPERIMENT_ROOT, "--allow_missing"], cwd=REPO_ROOT, env=env)
    run([sys.executable, SCRIPT_ROOT / "make_figures.py", "--manifest", SMOKE_MANIFEST, "--run_root", EXPERIMENT_ROOT], cwd=REPO_ROOT, env=env)
    run([sys.executable, SCRIPT_ROOT / "write_run_reports.py", "--run_root", EXPERIMENT_ROOT, "--smoke_manifest", SMOKE_MANIFEST, "--mini_manifest", MINI_MANIFEST], cwd=REPO_ROOT, env=env)


$ /usr/bin/python3 /content/FLUX-bss/bss_experiments/flux_fill_bss_bds_v1/scripts/run_manifest.py --manifest /content/FLUX-bss-Runs/flux_fill_bss_bds_v1/manifests/flux_fill_smoke_manifest.csv --run_root /content/FLUX-bss-Runs/flux_fill_bss_bds_v1 --drive_run_root /content/drive/MyDrive/Colab_Projects/FLUX-bss/runs/flux_fill_bss_bds_v1 --model_path /content/drive/MyDrive/Colab_Projects/FLUX-bss/models/FLUX.1-Fill-dev --resume --sync_drive true --dtype bfloat16
$ /usr/bin/python3 /content/FLUX-bss/bss_experiments/flux_fill_bss_bds_v1/scripts/validate_schedule.py --manifest /content/FLUX-bss-Runs/flux_fill_bss_bds_v1/manifests/flux_fill_smoke_manifest.csv --require_schedule_files
$ /usr/bin/python3 /content/FLUX-bss/bss_experiments/flux_fill_bss_bds_v1/scripts/compute_metrics_against_ref.py --manifest /content/FLUX-bss-Runs/flux_fill_bss_bds_v1/manifests/flux_fill_smoke_manifest.csv --run_root /content/FLUX-bss-Runs/flux_fill_bss_bds_v1 --allow_missing
$ /usr/bin/python3 /content/FLUX-bss

In [53]:
if RUN_MINI_SUITE:
    cmd = [
        sys.executable, SCRIPT_ROOT / "run_manifest.py",
        "--manifest", MINI_MANIFEST,
        "--run_root", EXPERIMENT_ROOT,
        "--drive_run_root", DRIVE_EXPERIMENT_ROOT,
        "--model_path", DRIVE_WEIGHTS_ROOT,
        "--resume",
        "--sync_drive", "true",
        "--dtype", DTYPE,
    ]
    if CPU_OFFLOAD:
        cmd.append("--cpu_offload")
    try:
        run(cmd, cwd=REPO_ROOT, env=env)
    except subprocess.CalledProcessError:
        show_manifest_failures(MINI_MANIFEST)
        raise
    run([sys.executable, SCRIPT_ROOT / "validate_schedule.py", "--manifest", MINI_MANIFEST, "--require_schedule_files"], cwd=REPO_ROOT, env=env)
    run([sys.executable, SCRIPT_ROOT / "compute_metrics_against_ref.py", "--manifest", MINI_MANIFEST, "--run_root", EXPERIMENT_ROOT, "--allow_missing"], cwd=REPO_ROOT, env=env)
    run([sys.executable, SCRIPT_ROOT / "compute_bds.py", "--run_root", EXPERIMENT_ROOT, "--allow_missing"], cwd=REPO_ROOT, env=env)
    run([sys.executable, SCRIPT_ROOT / "make_tables_flux_fill.py", "--run_root", EXPERIMENT_ROOT], cwd=REPO_ROOT, env=env)
    run([sys.executable, SCRIPT_ROOT / "make_figures.py", "--manifest", MINI_MANIFEST, "--run_root", EXPERIMENT_ROOT], cwd=REPO_ROOT, env=env)
    run([sys.executable, SCRIPT_ROOT / "write_run_reports.py", "--run_root", EXPERIMENT_ROOT, "--smoke_manifest", SMOKE_MANIFEST, "--mini_manifest", MINI_MANIFEST], cwd=REPO_ROOT, env=env)
    run([sys.executable, SCRIPT_ROOT / "write_final_report.py", "--run_root", EXPERIMENT_ROOT, "--notebook_path", REPO_ROOT / "notebooks/flux_fill_colab_bss_bds.ipynb"], cwd=REPO_ROOT, env=env)
else:
    print("RUN_MINI_SUITE is False. Review smoke outputs before enabling the mini-suite.")


$ /usr/bin/python3 /content/FLUX-bss/bss_experiments/flux_fill_bss_bds_v1/scripts/run_manifest.py --manifest /content/FLUX-bss-Runs/flux_fill_bss_bds_v1/manifests/flux_fill_mini_manifest.csv --run_root /content/FLUX-bss-Runs/flux_fill_bss_bds_v1 --drive_run_root /content/drive/MyDrive/Colab_Projects/FLUX-bss/runs/flux_fill_bss_bds_v1 --model_path /content/drive/MyDrive/Colab_Projects/FLUX-bss/models/FLUX.1-Fill-dev --resume --sync_drive true --dtype bfloat16
$ /usr/bin/python3 /content/FLUX-bss/bss_experiments/flux_fill_bss_bds_v1/scripts/validate_schedule.py --manifest /content/FLUX-bss-Runs/flux_fill_bss_bds_v1/manifests/flux_fill_mini_manifest.csv --require_schedule_files
$ /usr/bin/python3 /content/FLUX-bss/bss_experiments/flux_fill_bss_bds_v1/scripts/compute_metrics_against_ref.py --manifest /content/FLUX-bss-Runs/flux_fill_bss_bds_v1/manifests/flux_fill_mini_manifest.csv --run_root /content/FLUX-bss-Runs/flux_fill_bss_bds_v1 --allow_missing
$ /usr/bin/python3 /content/FLUX-bss/bs

In [54]:
if PRINT_FINAL_SUMMARY:
    import json
    import shutil
    from pathlib import Path

    import pandas as pd
    from IPython.display import Markdown, display

    if SYNC_FINAL_TO_DRIVE and Path("/content/drive").exists():
        DRIVE_EXPERIMENT_ROOT.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(EXPERIMENT_ROOT, DRIVE_EXPERIMENT_ROOT, dirs_exist_ok=True)
        print("Synced final artifacts to Drive:", DRIVE_EXPERIMENT_ROOT)

    final_report = EXPERIMENT_ROOT / "reports/FINAL_FLUX_FILL_BSS_BDS_REPORT.md"
    bds_report = EXPERIMENT_ROOT / "reports/03_bds_report.md"
    bds_row = EXPERIMENT_ROOT / "tables/cross_model_bds_row.csv"
    bds_table = EXPERIMENT_ROOT / "tables/tableA_flux_fill_bds_by_split.csv"
    mask_row = EXPERIMENT_ROOT / "tables/table_cross_model_same_compute_flux_fill_row.csv"
    full_row = EXPERIMENT_ROOT / "tables/table_cross_model_same_compute_flux_fill_row_full_rgb.csv"
    latex_row = EXPERIMENT_ROOT / "tables/table_cross_model_same_compute_flux_fill_row.tex"
    gains = EXPERIMENT_ROOT / "metrics/same_compute_gain_long.csv"
    metrics = EXPERIMENT_ROOT / "metrics/master_long_metrics.csv"
    side_by_side = EXPERIMENT_ROOT / "figures/side_by_side/index.html"

    def show_manifest_status(path, title):
        path = Path(path)
        print(f"\n=== {title} ===")
        print(path)
        if not path.exists():
            print("missing")
            return None
        df = pd.read_csv(path).fillna("")
        status = df["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="rows")
        display(status)
        failed = df[df["status"].astype(str).str.lower().eq("failed")]
        if not failed.empty:
            display(failed[["run_id", "case_id", "method", "error_message"]])
        return df

    def show_csv(path, title, max_rows=20):
        path = Path(path)
        print(f"\n=== {title} ===")
        print(path)
        if not path.exists():
            print("missing")
            return None
        df = pd.read_csv(path)
        display(df.head(max_rows))
        if len(df) > max_rows:
            print(f"... {len(df) - max_rows} more rows")
        return df

    def show_markdown(path, title, max_chars=12000):
        path = Path(path)
        print(f"\n=== {title} ===")
        print(path)
        if not path.exists():
            print("missing")
            return
        text = path.read_text(encoding="utf-8", errors="replace")
        if len(text) > max_chars:
            text = text[:max_chars] + "\n\n... truncated ..."
        display(Markdown(text))

    smoke_df = show_manifest_status(SMOKE_MANIFEST, "Smoke Manifest Status")
    mini_df = show_manifest_status(MINI_MANIFEST, "Mini-Suite Manifest Status")
    bds_df = show_csv(bds_row, "Cross-Model BDS Row")
    show_csv(mask_row, "Mask RGB-L1 Closure Row")
    show_csv(full_row, "Full RGB-L1 Closure Row")
    show_csv(bds_table, "BDS By Split")
    show_csv(gains, "Same-Compute Gain Long", max_rows=32)

    if bds_df is not None and not bds_df.empty and "final_verdict" in bds_df.columns:
        print("\nFINAL_VERDICT:", bds_df.iloc[0]["final_verdict"])

    print("\n=== Key Files ===")
    for label, result_path in [
        ("final_report", final_report),
        ("bds_report", bds_report),
        ("mask_row", mask_row),
        ("full_row", full_row),
        ("latex_row", latex_row),
        ("bds_table", bds_table),
        ("metrics", metrics),
        ("gains", gains),
        ("side_by_side", side_by_side),
        ("drive_mirror", DRIVE_EXPERIMENT_ROOT),
    ]:
        print(f"{label}: {result_path} {'[ok]' if Path(result_path).exists() else '[missing]'}")

    show_markdown(final_report, "Final Report")


Synced final artifacts to Drive: /content/drive/MyDrive/Colab_Projects/FLUX-bss/runs/flux_fill_bss_bds_v1

=== Smoke Manifest Status ===
/content/FLUX-bss-Runs/flux_fill_bss_bds_v1/manifests/flux_fill_smoke_manifest.csv


,status,rows
0,pending,4



=== Mini-Suite Manifest Status ===
/content/FLUX-bss-Runs/flux_fill_bss_bds_v1/manifests/flux_fill_mini_manifest.csv


,status,rows
0,pending,32



=== Cross-Model BDS Row ===
/content/FLUX-bss-Runs/flux_fill_bss_bds_v1/tables/cross_model_bds_row.csv


...

,split_name,tset_name,calibration_cases,calibration_observations,calibration_mean_mask_gain,calibration_mask_lcb95,calibration_mean_full_gain,mean_preservation_delta,preservation_guard_ok,predicted_verdict,holdout_cases,holdout_observations,holdout_mean_mask_gain,holdout_win_rate,notes
0,first_half_split,low,2,2,0.368373,0.091644,0.327998,0.000182,True,Provisional Low-only,2,2,0.152887,1,Tset={10}; provisional_4case
1,first_half_split,all,2,6,0.101826,-0.071485,0.093487,0.000212,True,Provisional Low-only,2,6,0.112968,1,"Tset={10,20,40}; provisional_4case"
2,alternating_split,low,2,2,0.136367,0.091644,0.152297,0.000002,True,Provisional Low-only,2,2,0.384893,1,Tset={10}; provisional_4case
3,alternating_split,all,2,6,0.024692,-0.073718,0.041953,0.000052,True,Provisional Low-only,2,6,0.190101,1,"Tset={10,20,40}; provisional_4case"
4,random_split_seed0,low,2,2,0.136367,0.091644,0.152297,0.000002,True,Provisional Low-only,2,2,0.384893,1,Tset={10}; provisional_4case
5,random_split_seed0,all,2,6,0.024692,-0.073757,0.041953,0.000052,True,Provisional Low-only,2,6,0.190101,1,"Tset={10,20,40}; provisional_4case"



=== Same-Compute Gain Long ===
/content/FLUX-bss-Runs/flux_fill_bss_bds_v1/metrics/same_compute_gain_long.csv


,case_id,category,actual_nfe,compute_fraction,uniform_method,bss_method,uniform_mask_rgb_l1_closure,bss_mask_rgb_l1_closure,mask_rgb_l1_closure_gain,uniform_full_rgb_l1_closure,bss_full_rgb_l1_closure,full_rgb_l1_closure_gain,preservation_delta_bss_minus_uniform,bss_win_mask,bss_win_full,reference_method,reference_nfe
0,case001,object_replacement_on_table,10,0.2,uniform10,bss10,0.140037,0.231681,0.091644,0.117321,0.232201,0.114880,-0.000203,True,True,reference_uniform50,50
1,case001,object_replacement_on_table,20,0.4,uniform20,bss20,0.503208,0.332293,-0.170915,0.492612,0.382337,-0.110275,-0.000135,False,False,reference_uniform50,50
2,case001,object_replacement_on_table,40,0.8,uniform40,bss40,0.744803,0.593244,-0.151559,0.752928,0.643718,-0.109211,0.000115,False,False,reference_uniform50,50
3,case002,indoor_object_remove_replace,10,0.2,uniform10,bss10,-0.299526,0.345575,0.645102,-0.188473,0.352643,0.541116,0.000566,True,True,reference_uniform50,50
4,case002,indoor_object_remove_replace,20,0.4,uniform20,bss20,0.462448,0.541192,0.078745,0.473407,0.582832,0.109425,0.000332,True,True,reference_uniform50,50
5,case002,indoor_object_remove_replace,40,0.8,uniform40,bss40,0.736100,0.854038,0.117938,0.747729,0.762717,0.014988,0.000595,True,True,reference_uniform50,50
6,case003,outdoor_missing_region_fill,10,0.2,uniform10,bss10,0.052031,0.233120,0.181089,0.059294,0.249008,0.189714,0.000207,True,True,reference_uniform50,50
7,case003,outdoor_missing_region_fill,20,0.4,uniform20,bss20,0.408687,0.573415,0.164728,0.479392,0.640008,0.160616,0.000119,True,True,reference_uniform50,50
8,case003,outdoor_missing_region_fill,40,0.8,uniform40,bss40,0.766627,0.799795,0.033168,0.770714,0.776707,0.005993,0.000210,True,True,reference_uniform50,50
9,case004,local_texture_lighting_edit,10,0.2,uniform10,bss10,0.227181,0.351865,0.124684,0.218793,0.342682,0.123889,-0.000008,True,True,reference_uniform50,50



FINAL_VERDICT: Provisional Low-only

=== Key Files ===
final_report: /content/FLUX-bss-Runs/flux_fill_bss_bds_v1/reports/FINAL_FLUX_FILL_BSS_BDS_REPORT.md [ok]
bds_report: /content/FLUX-bss-Runs/flux_fill_bss_bds_v1/reports/03_bds_report.md [ok]
mask_row: /content/FLUX-bss-Runs/flux_fill_bss_bds_v1/tables/table_cross_model_same_compute_flux_fill_row.csv [ok]
full_row: /content/FLUX-bss-Runs/flux_fill_bss_bds_v1/tables/table_cross_model_same_compute_flux_fill_row_full_rgb.csv [ok]
latex_row: /content/FLUX-bss-Runs/flux_fill_bss_bds_v1/tables/table_cross_model_same_compute_flux_fill_row.tex [ok]
bds_table: /content/FLUX-bss-Runs/flux_fill_bss_bds_v1/tables/tableA_flux_fill_bds_by_split.csv [ok]
metrics: /content/FLUX-bss-Runs/flux_fill_bss_bds_v1/metrics/master_long_metrics.csv [ok]
gains: /content/FLUX-bss-Runs/flux_fill_bss_bds_v1/metrics/same_compute_gain_long.csv [ok]
side_by_side: /content/FLUX-bss-Runs/flux_fill_bss_bds_v1/figures/side_by_side/index.html [ok]
drive_mirror: /conten

# Final FLUX Fill BSS/BDS Report

## 1. Purpose

Condition-anchoring smoke after Sana-0.6B text-only Reject. This is not a FLUX SOTA benchmark and not a universal BSS claim.

## 2. Model / License / Hardware Audit

- audit report: `/content/FLUX-bss-Runs/flux_fill_bss_bds_v1/reports/00_repo_model_hardware_audit.md`
- Drive weight path: `/content/drive/MyDrive/Colab_Projects/FLUX-bss/models/FLUX.1-Fill-dev`

## 3. Notebook

- Colab notebook path: `/content/FLUX-bss/notebooks/flux_fill_colab_bss_bds.ipynb`

## 4. Source Image / Mask Suite

- asset manifest: `/content/FLUX-bss-Runs/flux_fill_bss_bds_v1/assets/asset_manifest.md`

## 5. BSS Schedule Implementation

BSS constructs a base schedule with T-2 scheduler coordinates, splits the first and last intervals, and passes custom `sigmas` to `FluxFillPipeline` when supported. If the installed diffusers pipeline does not support custom `sigmas`, this scaffold stops instead of faking BSS.

## 6. Smoke Result

- smoke report: `/content/FLUX-bss-Runs/flux_fill_bss_bds_v1/reports/01_flux_fill_smoke_report.md`

## 7. Mini-Suite Result

- mini-suite report: `/content/FLUX-bss-Runs/flux_fill_bss_bds_v1/reports/02_mini_suite_run_report.md`

## 8. Same-Compute Closure Tables

- mask-region row: `/content/FLUX-bss-Runs/flux_fill_bss_bds_v1/tables/table_cross_model_same_compute_flux_fill_row.csv`
- full-image row: `/content/FLUX-bss-Runs/flux_fill_bss_bds_v1/tables/table_cross_model_same_compute_flux_fill_row_full_rgb.csv`
- LaTeX row: `/content/FLUX-bss-Runs/flux_fill_bss_bds_v1/tables/table_cross_model_same_compute_flux_fill_row.tex`

## 9. Unmasked Preservation

Unmasked preservation is reported as BSS minus same-NFE uniform unmasked RGB-L1 to source. Positive values trigger the preservation guard if they exceed tolerance.

## 10. BDS Calibration / Holdout

- BDS table: `/content/FLUX-bss-Runs/flux_fill_bss_bds_v1/tables/tableA_flux_fill_bds_by_split.csv`
- BDS report: `/content/FLUX-bss-Runs/flux_fill_bss_bds_v1/reports/03_bds_report.md`

## 11. Figures

- mask closure figure: `/content/FLUX-bss-Runs/flux_fill_bss_bds_v1/figures/compute_quality_mask_rgb_closure.png`
- full closure figure: `/content/FLUX-bss-Runs/flux_fill_bss_bds_v1/figures/compute_quality_full_rgb_closure.png`
- side-by-side index: `/content/FLUX-bss-Runs/flux_fill_bss_bds_v1/figures/side_by_side/index.html`

## 12. Final Verdict

`Provisional Low-only`

## 13. Caveats

- FLUX.1 Fill-dev is gated and uses a non-commercial dev license.
- Fixed synthetic/public assets are a smoke suite, not a benchmark.
- Reference is uniform50, not ground truth.
- NFE is used as the compute proxy.
- Fill/inpainting metrics depend on mask definition.
- The model card describes guidance distillation; do not label this as path-consistency distillation unless separately audited.

## 14. Next Steps

- Expand to 8 cases if 4-case smoke/mini trends are promising.
- Try FLUX.1 Canny-dev after fill tools pass.
- Try Qwen-Image-Edit only after FLUX tooling is stable.
